# 宅建士 Phase 4-5: QLoRA ファインチューニング + FT後評価

**目的**: ベースモデル `Qwen/Qwen2.5-7B-Instruct` を train 100問でQLoRAファインチューニングし、その後同じ eval 50問で正答率を測定する。FT前ベースライン (`results/baseline_eval.json`) と FT後 (`results/ft_eval.json`) を比較するためのデータを生成する。

**重要な前提（`docs/freeze-spec.md` §5 / §6 完全準拠）**:
- 学習: QLoRA (4bit nf4) + Unsloth、`r=16`, `alpha=32`, `dropout=0.05`, `epoch=2`, `lr=2e-4`, `batch=2`, `grad_accum=4`, `max_seq=2048`, `seed=42`
- LoRA target_modules: `q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj`
- 推論・採点ロジックは `baseline_eval.ipynb` と **完全同一**（system promptも一字一句一致、`temperature=0.0`, `max_new_tokens=8`, `do_sample=False`, seed=42、抽出は最初の `[1-4]`）
- ロス計算は **assistant 部分のみ**（`train_on_responses_only` ヘルパ使用）
- Drive のチェックポイントパス: `/content/drive/MyDrive/construction-llm-ft/checkpoints/takken-qwen25-7b/`
- `save_steps=50`, `save_total_limit=2`

**実行環境**:
- Colab **GPU T4 (16GB)** 必須（A100/V100 でも可、CPU不可）
- 想定実行時間: T4 で **約 1.5〜3 時間**（依存導入 5〜10分 + モデルDL 5〜10分 + 学習 1〜2時間 + 評価 10〜20分）
- VRAM: Unsloth 4bit Qwen2.5-7B + LoRA r=16 で約 8〜10GB（T4 16GB で余裕）
- MacBook を閉じても Mac Mini 上の Chrome タブで Colab セッションは継続する

**実行手順**:
1. ランタイムを GPU T4 に設定（メニュー: ランタイム > ランタイムのタイプを変更 > T4 GPU）
2. ランタイム > すべてのセルを実行
3. 完走すると `/content/drive/MyDrive/construction-llm-ft/results/ft_eval.json` に結果が保存される
4. ローカル/Mac Mini 側で取り込む（`results/ft_eval.json` を Drive→repo に手動 cp、もしくは同期済みなら何もしない）
5. その後 `scripts/plot_score_comparison.py`（後続タスク）で baseline と FT の比較グラフを生成

**禁止事項**（CLAUDE.md 公開方針）:
- `push_to_hub` などモデル公開系の呼び出しは行わない
- `huggingface_hub.login()` は呼ばない（Qwen2.5 は anonymous DL 可能）
- HF_TOKEN 参照なし

**Checkpoint 復帰**: 万一中断したら同じノートを再実行し、学習セルを `trainer.train(resume_from_checkpoint=True)` に書き換えれば再開可能（`save_steps=50` で Drive に保存されている）。


## 1. GPU 確認

`Qwen2.5-7B` を 4bit QLoRA 学習するには GPU が必須です。CPU/TPU では動作しません。


In [ ]:
!nvidia-smi

In [ ]:
# Python 側からも GPU を確認
try:
    import torch
    has_cuda = torch.cuda.is_available()
    print(f"torch.cuda.is_available(): {has_cuda}")
    if has_cuda:
        print(f"device count: {torch.cuda.device_count()}")
        print(f"device 0: {torch.cuda.get_device_name(0)}")
        print(f"CUDA version: {torch.version.cuda}")
    else:
        print("[WARN] CUDA が利用できません。ランタイム > ランタイムのタイプを変更 > T4 GPU を選択してください。")
except ImportError:
    print("torch 未インストール。次のセルで依存をインストールします。")


## 2. 依存ライブラリのインストール

- **Unsloth**: T4 最適化 LoRA トレーニング（素の transformers の約 2 倍速）
- **trl<0.9.0**: Unsloth と互換性のある SFTTrainer 系列
- **peft / accelerate / bitsandbytes**: LoRA + 4bit 量子化

`pip install` は 5〜10 分かかります。Unsloth のビルドが走るとさらに時間がかかる場合あり。


In [ ]:
!pip install -q --upgrade pip
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl<0.9.0" peft accelerate bitsandbytes


## 3. Google Drive のマウント

train/eval JSONL の読み込み、checkpoint 保存、結果保存に Drive を使います。
プロジェクトフォルダ: `/content/drive/MyDrive/construction-llm-ft/`

事前に Mac Mini 側から下記が同期されている必要があります:
- `data/jsonl/train/takken_train.jsonl` (100問想定)
- `data/jsonl/eval/takken_eval.jsonl` (50問)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os

WORK_DIR = '/content/drive/MyDrive/construction-llm-ft'
assert os.path.isdir(WORK_DIR), f"WORK_DIR not found: {WORK_DIR}. Mac Mini側で Drive 同期されているか確認してください。"
os.chdir(WORK_DIR)
print('cwd:', os.getcwd())

TRAIN_PATH = os.path.join(WORK_DIR, 'data/jsonl/train/takken_train.jsonl')
EVAL_PATH = os.path.join(WORK_DIR, 'data/jsonl/eval/takken_eval.jsonl')
CHECKPOINT_DIR = os.path.join(WORK_DIR, 'checkpoints/takken-qwen25-7b')
FINAL_ADAPTER_DIR = os.path.join(CHECKPOINT_DIR, 'final')
RESULTS_DIR = os.path.join(WORK_DIR, 'results')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

assert os.path.isfile(TRAIN_PATH), f"train JSONL が見つかりません: {TRAIN_PATH}"
assert os.path.isfile(EVAL_PATH), f"eval JSONL が見つかりません: {EVAL_PATH}"
print('train path     :', TRAIN_PATH)
print('eval path      :', EVAL_PATH)
print('checkpoint dir :', CHECKPOINT_DIR)
print('final dir      :', FINAL_ADAPTER_DIR)
print('results dir    :', RESULTS_DIR)


## 4. 学習データの読み込み

`takken_train.jsonl` を読み込んで件数を確認。期待値: 100 件（`docs/freeze-spec.md` §1 の train、PoC 暫定スコープ）。
各レコードは `{id, instruction, input, output}` 形式。


In [ ]:
import json

train_records = []
with open(TRAIN_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        train_records.append(json.loads(line))

print(f"loaded train: {len(train_records)} records")
assert len(train_records) == 100, f"期待 100 件、実際 {len(train_records)} 件"

# サンプル表示
sample = train_records[0]
print('keys :', list(sample.keys()))
print('id   :', sample['id'])
print('gold :', sample['output'])
print('input head:', sample['input'][:120], '...')


## 5. モデルのロード（Unsloth FastLanguageModel, 4bit nf4）

- `Qwen/Qwen2.5-7B-Instruct` を 4bit nf4 でロード（anonymous DL、HF_TOKEN 不要）
- `max_seq_length=2048`、`dtype=None` で T4 に自動最適化
- 初回ロードはモデル DL で 5〜10 分かかります


In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

# pad_token を eos に揃える
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("model loaded.")
print("model class:", type(model).__name__)


## 6. LoRA アダプタの適用

`docs/freeze-spec.md` §5 のハイパラ:
- `r=16`, `alpha=32`, `dropout=0.05`
- target_modules: `q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj`
- Unsloth の gradient checkpointing で VRAM 節約
- `random_state=42`


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
)

# 学習可能パラメータ数を表示
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"trainable params: {trainable_params:,}")
print(f"total params    : {total_params:,}")
print(f"trainable ratio : {trainable_params / total_params:.4%}")


## 7. 学習データの整形（Qwen2.5 ChatML 形式）

`tokenizer.apply_chat_template` で system / user / assistant の 3 ロールを構築。
- `add_generation_prompt=False`（assistant 応答も含めて学習対象テキストにする）
- system プロンプトは `baseline_eval.ipynb` および `docs/freeze-spec.md` §2 と**一字一句一致**

ロスマスクは後段 `train_on_responses_only` で assistant 部分のみに限定するので、ここではテキスト全体を入れる。


In [ ]:
from datasets import Dataset

# システムプロンプト（baseline_eval.ipynb と完全一致、freeze-spec.md §2）
SYSTEM_PROMPT = "あなたは宅地建物取引士資格試験の問題に正確に回答するアシスタントです。最も適切な選択肢の番号(1〜4)を1つだけ半角数字で答え、番号以外は出力してはいけません。"


def format_record(rec):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": rec["input"]},
        {"role": "assistant", "content": str(rec["output"])},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}


train_dataset = Dataset.from_list(train_records).map(format_record)
print(f"train_dataset size: {len(train_dataset)}")
print("---")
print("first record text (head 600 chars):")
print(train_dataset[0]["text"][:600])
print("---")
print("first record text (tail 200 chars):")
print(train_dataset[0]["text"][-200:])


## 8. SFTTrainer の構成

`docs/freeze-spec.md` §5 のハイパラを完全反映:
- `per_device_train_batch_size=2`, `gradient_accumulation_steps=4`（実効バッチ 8）
- `num_train_epochs=2`（100 問のため過学習回避、ユーザー指示確定）
- `lr=2e-4`, scheduler=cosine, `warmup_ratio=0.03`
- `optim="adamw_8bit"`（QLoRA 標準）
- `save_strategy="steps"`, `save_steps=50`, `save_total_limit=2`
- `seed=42`

その後 `train_on_responses_only` で **assistant 部分のみ** をロス計算対象にマスク。
`instruction_part` / `response_part` は Qwen2.5 ChatML のヘッダ文字列。


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_ratio=0.03,
    num_train_epochs=2,
    learning_rate=2e-4,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=5,
    optim="adamw_8bit",
    weight_decay=0.0,
    lr_scheduler_type="cosine",
    seed=42,
    output_dir=CHECKPOINT_DIR,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    args=training_args,
)

# assistant 応答部分のみをロス計算対象にマスク
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

print("trainer ready.")
print(f"effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"num_train_epochs    : {training_args.num_train_epochs}")
print(f"checkpoint dir      : {training_args.output_dir}")


## 9. 学習実行

T4 で約 **1〜2 時間**。`logging_steps=5` ごとにロスが表示されます。
中断した場合、同じノートをもう一度上から実行すれば、`trainer.train(resume_from_checkpoint=True)` に書き換えて再開できます（`save_steps=50` で Drive に保存）。


In [ ]:
trainer_stats = trainer.train()

print("---")
print("training metrics:")
try:
    print(f"final loss     : {trainer_stats.training_loss:.6f}")
except Exception as e:
    print(f"(could not read training_loss: {e})")
print(f"global_step    : {trainer.state.global_step}")
print(f"epoch          : {trainer.state.epoch}")


## 10. 最終アダプタの保存

LoRA アダプタを Drive に保存。`push_to_hub` 系は **一切呼ばない**（CLAUDE.md 公開方針）。


In [ ]:
model.save_pretrained(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)
print(f"saved final adapter: {FINAL_ADAPTER_DIR}")

# 保存ファイル一覧
for name in sorted(os.listdir(FINAL_ADAPTER_DIR)):
    full = os.path.join(FINAL_ADAPTER_DIR, name)
    size = os.path.getsize(full) if os.path.isfile(full) else 0
    print(f"  {name}  ({size:,} bytes)")


## 11. FT後評価（baseline_eval と完全同一仕様）

`docs/freeze-spec.md` §6 比較成立条件:
- 同一 eval セット（2025年度50問）
- 同一プロンプトテンプレート（§2）
- 同一生成パラメータ（§3）: `temperature=0.0`, `do_sample=False`, `max_new_tokens=8`, `top_p=1.0`, `repetition_penalty=1.0`, `seed=42`
- 同一採点ロジック（§4）: 最初の `[1-4]` を抽出、なければ `unparseable`
- 同一ベースモデル / 同一 4bit 量子化（学習時もFT後推論時も nf4-4bit）

学習からシームレスに切り替えるため、`FastLanguageModel.for_inference(model)` で推論モードに移行（LoRAは適用状態のまま）。


In [ ]:
import re
import datetime as _dt
from tqdm.auto import tqdm

# 推論モードへ切替（LoRA は適用状態のまま）
FastLanguageModel.for_inference(model)

# eval 読み込み
eval_records = []
with open(EVAL_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        eval_records.append(json.loads(line))

print(f"loaded eval: {len(eval_records)} records")
assert len(eval_records) == 50, f"期待 50 件、実際 {len(eval_records)} 件"


# シード固定（temperature=0 なので決定論的だが念のため明示。baseline と一致）
SEED = 42
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


@torch.no_grad()
def generate_answer(user_content: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=8,
        do_sample=False,
        temperature=0.0,
        top_p=1.0,
        repetition_penalty=1.0,
        pad_token_id=tokenizer.eos_token_id,
    )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


def extract_answer(raw_output: str) -> str:
    # freeze-spec.md §4 と完全一致
    stripped = raw_output.strip()
    m = re.search(r"[1-4]", stripped)
    return m.group(0) if m else "unparseable"


# 動作確認: 1 問だけ流す
_sample = eval_records[0]
_raw = generate_answer(_sample["input"])
_pred = extract_answer(_raw)
print(f"sample id  : {_sample['id']}")
print(f"raw output : {_raw!r}")
print(f"pred       : {_pred}")
print(f"gold       : {_sample['output']}")
print(f"correct?   : {_pred == _sample['output']}")


## 12. 評価ループ（50問）

`tqdm` で進捗表示。温度 0 の決定論的生成なので 1 回だけで十分。
T4 で約 5〜10 分。


In [ ]:
per_question = []
num_correct = 0

for rec in tqdm(eval_records, desc="evaluating (FT)"):
    qid = rec["id"]
    gold = str(rec["output"]).strip()
    user_content = rec["input"]

    raw = generate_answer(user_content)
    pred = extract_answer(raw)
    is_correct = (pred == gold)
    if is_correct:
        num_correct += 1

    per_question.append({
        "id": qid,
        "pred": pred,
        "gold": gold,
        "correct": is_correct,
        "raw": raw,
    })

num_total = len(eval_records)
accuracy = num_correct / num_total if num_total > 0 else 0.0
print(f"\nFT accuracy: {num_correct}/{num_total} = {accuracy:.4f}")


## 13. 結果の保存

`docs/freeze-spec.md` §7 のフォーマットに `baseline_eval.json` と互換するよう、`date`, `score`, `config` を統合して保存。
保存先: `<WORK_DIR>/results/ft_eval.json`

`adapter` フィールドに checkpoint パスを記録する（baseline では `null`）。


In [ ]:
now = _dt.datetime.now(_dt.timezone(_dt.timedelta(hours=9)))  # JST
run_id = f"ft_{now.strftime('%Y%m%d')}_e2_lr2e-4"
date_str = now.strftime("%Y-%m-%d")
timestamp = now.isoformat(timespec="seconds")

# adapter は repo 相対パス表記に揃える
ADAPTER_REL = "checkpoints/takken-qwen25-7b/final"

result_obj = {
    "run_id": run_id,
    "date": date_str,
    "model": MODEL_NAME,
    "adapter": ADAPTER_REL,
    "eval_set": "takken_2025",
    "num_total": num_total,
    "num_correct": num_correct,
    "accuracy": accuracy,
    "score": f"{num_correct}/{num_total}",
    "per_question": per_question,
    "config": {
        "quantization": "nf4-4bit",
        "compute_dtype": "auto(unsloth)",
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05,
        "lora_target_modules": [
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        "epoch": 2,
        "learning_rate": 2e-4,
        "batch_size": 2,
        "grad_accum": 4,
        "max_seq_length": MAX_SEQ_LENGTH,
        "optimizer": "adamw_8bit",
        "scheduler": "cosine",
        "warmup_ratio": 0.03,
        "seed": SEED,
        "train_size": len(train_records),
        "temperature": 0.0,
        "top_p": 1.0,
        "do_sample": False,
        "max_new_tokens": 8,
        "repetition_penalty": 1.0,
    },
    "timestamp": timestamp,
}

out_path = os.path.join(RESULTS_DIR, "ft_eval.json")
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(result_obj, f, ensure_ascii=False, indent=2)

print(f"saved: {out_path}")
print(f"FT accuracy: {num_correct}/{num_total} = {accuracy:.4f}")


## 14. 完了後の手順

- このノートは FT 実行 + FT後評価を行う
- 結果ファイル `results/ft_eval.json` は Google Drive 側に保存される
- アダプタ `checkpoints/takken-qwen25-7b/final/` は Drive 側に保存される（merge 済みではない、LoRA だけ）
- Mac Mini で結果を取り込みたい場合:
  ```bash
  # Mac Mini 側（Driveが手動マウントの場合）
  cd ~/construction-llm-ft
  cp "/path/to/Drive/construction-llm-ft/results/ft_eval.json" ./results/
  git add results/ft_eval.json WORK_LOG.md
  git commit -m "Phase 4-5: FT + FT-eval result"
  git push
  ```
  ※ checkpoint 本体（数百MB〜数GB）は repo に含めない。`.gitignore` で `checkpoints/` を除外しておくこと
- 次のステップ: `scripts/plot_score_comparison.py` を Mac Mini 側で実行し、`results/baseline_eval.json` と `results/ft_eval.json` から比較グラフを生成
- 比較条件は変えないこと（`docs/freeze-spec.md` §6）

**禁止事項の再確認**:
- `model.push_to_hub(...)` / `trainer.push_to_hub(...)` は呼ばない
- HF Hub へのアップロード関連スクリプトは作らない
- private repo のままにしておく
